In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 0 — NAR Surgical Patch (Option B, PR #548)
# Patches the installed AR casanovo classes in-process at class level.
# Must run FIRST. No kernel restart needed — patches persist in memory.
#
# What the AR install has:          What we patch it to:
#   PeptideDecoder  → no embed()      → embed() with all-False tgt_mask
#   Spec2Pep        → GT tokens       → zero tokens in _forward_step
#   Spec2Pep.forward→ beam_search     → calls _forward_step
# ═══════════════════════════════════════════════════════════════════════
import subprocess, sys, warnings, inspect
warnings.filterwarnings('ignore')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'casanovo>=5.0.0', 'pyteomics', 'lxml', 'remotezip', 'appdirs'], check=True)

import torch
from casanovo.denovo.transformers import PeptideDecoder
from casanovo.denovo.model import Spec2Pep

# ── Capture the AR (inherited) embed BEFORE we overwrite it ──────────
# In AR casanovo, PeptideDecoder has no embed() → it inherits from
# AnalyteTransformerDecoder. Capturing here gives us the parent's embed
# to call inside our NAR version (equivalent to super().embed(…)).
_ar_embed_original = PeptideDecoder.embed

# ─────────────────────────────────────────────────────────────────────
# PATCH 1 — PeptideDecoder.embed
# Source: PR #548 transformers.py
# Replaces the causal upper-triangular tgt_mask the parent builds with
# an all-False (L×L) mask, allowing every position to attend to every
# other position — the core of non-autoregressive parallel decoding.
# ─────────────────────────────────────────────────────────────────────
def _nar_embed(self, tokens, *args,
               memory,
               memory_key_padding_mask=None,
               memory_mask=None,
               tgt_mask=None,
               **kwargs):
    if tokens is None:
        # Match base-class convention: empty (1, 0) tensor on correct device
        tokens = torch.tensor([[]], dtype=torch.float,
                               device=next(self.parameters()).device)
    L = tokens.shape[1] + 1          # +1 for the prepended global token
    tgt_mask = torch.zeros((L, L), dtype=torch.bool, device=tokens.device)
    return _ar_embed_original(
        self, tokens, *args,
        memory=memory,
        memory_key_padding_mask=memory_key_padding_mask,
        memory_mask=memory_mask,
        tgt_mask=tgt_mask,
        **kwargs,
    )

PeptideDecoder.embed = _nar_embed

# ─────────────────────────────────────────────────────────────────────
# PATCH 2 — Spec2Pep._forward_step
# Source: PR #548 model.py
# AR version feeds ground-truth tokens (teacher forcing).
# NAR version feeds an all-zero tensor so the decoder predicts ALL
# positions in a single parallel pass, with no access to prior outputs.
# ─────────────────────────────────────────────────────────────────────
def _nar_forward_step(self, batch):
    mzs, ints, precursors, seqs = self._process_batch(batch)
    dev = self.device
    # Explicit .to(dev) makes direct profiling calls safe (Lightning
    # normally handles this, but we call the model outside a Trainer).
    mzs       = mzs.to(dev)
    ints      = ints.to(dev)
    precursors = precursors.to(dev)

    memories, mem_masks = self.encoder(mzs, ints)

    if seqs is not None:                    # training: match GT length
        zero_tokens = torch.zeros_like(seqs.to(dev))
    else:                                   # inference: full max length
        zero_tokens = torch.zeros(
            (mzs.shape[0], self.max_peptide_len),
            dtype=torch.long, device=dev,
        )
    scores = self.decoder(
        tokens=zero_tokens,
        memory=memories,
        memory_key_padding_mask=mem_masks,
        precursors=precursors,
    )
    return scores, seqs

Spec2Pep._forward_step = _nar_forward_step

# ─────────────────────────────────────────────────────────────────────
# PATCH 3 — Spec2Pep.forward
# Source: PR #548 model.py
# AR version called beam_search_decode; NAR delegates to _forward_step.
# ─────────────────────────────────────────────────────────────────────
def _nar_forward(self, batch):
    return self._forward_step(batch)

Spec2Pep.forward = _nar_forward

# ── Verification ──────────────────────────────────────────────────────
try:
    _ok_embed = 'tgt_mask = torch.zeros' in inspect.getsource(_nar_embed)
    _ok_fwd   = 'zero_tokens' in inspect.getsource(_nar_forward_step)
except (OSError, TypeError):
    # Fallback: identity check (works even when source is unavailable)
    _ok_embed = True
    _ok_fwd   = True

_ok_id_embed = (PeptideDecoder.embed  is _nar_embed)
_ok_id_fwd   = (Spec2Pep._forward_step is _nar_forward_step)
_ok_id_fwd2  = (Spec2Pep.forward       is _nar_forward)

print('\n── NAR Patch Status ──────────────────────────────────────────')
print(f'  PeptideDecoder.embed  patched (identity)  : {"✓" if _ok_id_embed else "✗ FAILED"}')
print(f'  Spec2Pep._forward_step patched (identity) : {"✓" if _ok_id_fwd   else "✗ FAILED"}')
print(f'  Spec2Pep.forward       patched (identity) : {"✓" if _ok_id_fwd2  else "✗ FAILED"}')
print(f'  all-False tgt_mask in embed source        : {"✓" if _ok_embed    else "~ (source check skipped)"}')
print(f'  zero_tokens in _forward_step source       : {"✓" if _ok_fwd      else "~ (source check skipped)"}')
print('──────────────────────────────────────────────────────────────')

if not (_ok_id_embed and _ok_id_fwd and _ok_id_fwd2):
    raise RuntimeError('One or more NAR patches failed — check errors above.')

print('\nNAR patches applied ✓  No kernel restart needed.')
print('Continue → run Cell 1 (Setup) next.')


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip



── NAR Patch Status ──────────────────────────────────────────
  PeptideDecoder.embed  patched (identity)  : ✓
  Spec2Pep._forward_step patched (identity) : ✓
  Spec2Pep.forward       patched (identity) : ✓
  all-False tgt_mask in embed source        : ✓
  zero_tokens in _forward_step source       : ✓
──────────────────────────────────────────────────────────────

NAR patches applied ✓  No kernel restart needed.
Continue → run Cell 1 (Setup) next.


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1 — Setup
# ═══════════════════════════════════════════════════════════════════════
import subprocess, sys, os, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import datetime
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch._dynamo
from torch.profiler import profile, ProfilerActivity, schedule, record_function
from pathlib import Path
from tqdm import tqdm

torch.manual_seed(42)

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_NAME   = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'
TOTAL_VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == 'cuda' else 0

N_PEAKS = 150     # FIXED peak budget — required for CUDA Graphs (static shapes
                  # only). Real spectra with more peaks are reduced to their
                  # N_PEAKS most-intense peaks (see pad_or_select_to_fixed_peaks
                  # below); spectra with fewer peaks are zero-padded. This SAME
                  # fixed shape is used for both the baseline (eager) and the
                  # torch.compile + CUDA Graph runs, so the comparison isolates
                  # only the optimization, not a change in input fidelity.
AVG_PEP = 12

# ── NAR profiling constants ─────────────────────────────────────────
N_SUBSET         = 6000
N_TIMING_SPECTRA = 5000
BATCH_SIZES      = [1, 8, 32, 128, 512]
N_WARMUP_BATCHES = 15     # warm-up batches before timing each batch size.
                          # For the compiled path this warm-up window is where
                          # the (slow, one-time) torch.compile trace + CUDA Graph
                          # capture happens for that batch size's fixed shape.

PROF_WARMUP = 20
PROF_ACTIVE = 50          # ≥50 real spectra profiled in detail

# ── torch.compile + CUDA Graphs settings ─────────────────────────────
# mode='reduce-overhead' is PyTorch's official combined torch.compile +
# CUDA Graph mechanism: TorchInductor compiles each submodule once per
# distinct input shape, then replays the captured CUDA Graph on every
# later call with that shape — eliminating per-kernel CPU dispatch.
# Docs:
#   https://docs.pytorch.org/docs/stable/generated/torch.compile.html
#   https://docs.pytorch.org/tutorials/intermediate/torch_compile_tutorial.html
#   https://docs.pytorch.org/docs/stable/notes/cuda.html#cuda-graphs
COMPILE_MODE = 'reduce-overhead'

# Raise the recompilation cache limit: we intentionally trigger one
# compile + graph-capture per (module, batch_size) combination — up to
# 2 modules × 5 batch sizes = 10 distinct shapes. Default cache_size_limit
# (8) would silently fall back to eager for any shape past the limit.
torch._dynamo.config.cache_size_limit = 32

def _sync():
    if DEVICE == 'cuda':
        torch.cuda.synchronize()

def pad_or_select_to_fixed_peaks(mzs, ints, n_peaks=N_PEAKS):
    """
    Coerce a (batch, L) mzs/ints pair — already collated by the DataLoader,
    where L is the longest real-peak count within that batch — into a FIXED
    (batch, n_peaks) shape. CUDA Graphs require every input tensor to have
    an identical shape on every replay; this is what makes that possible
    for spectra of widely varying peak counts (6 to 950 in this dataset).

      L == n_peaks : returned unchanged
      L <  n_peaks : zero-padded on the right (same convention the model's
                     own batch-collation already uses — zero intensity
                     marks a non-real peak)
      L >  n_peaks : reduced to the n_peaks MOST INTENSE peaks per spectrum
                     (not naive head-truncation), re-sorted by m/z ascending
                     to preserve the canonical peak ordering the encoder
                     was trained on

    Returns (mzs_out, ints_out, n_spectra_truncated_this_call).
    """
    bs, L = mzs.shape
    if L == n_peaks:
        return mzs, ints, 0
    if L < n_peaks:
        pad = n_peaks - L
        mzs_out  = torch.nn.functional.pad(mzs,  (0, pad), value=0.0)
        ints_out = torch.nn.functional.pad(ints, (0, pad), value=0.0)
        return mzs_out, ints_out, 0
    # L > n_peaks: keep top-n_peaks most intense peaks, re-sorted by m/z
    _, topk_idx = torch.topk(ints, k=n_peaks, dim=1)
    mz_at_topk  = torch.gather(mzs, 1, topk_idx)
    sort_idx    = torch.argsort(mz_at_topk, dim=1)
    final_idx   = torch.gather(topk_idx, 1, sort_idx)
    mzs_out  = torch.gather(mzs,  1, final_idx)
    ints_out = torch.gather(ints, 1, final_idx)
    return mzs_out, ints_out, bs

print(f'Device : {DEVICE} | GPU: {GPU_NAME} | VRAM: {TOTAL_VRAM:.1f} GB')
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')
print(f'N_SUBSET={N_SUBSET} | N_TIMING_SPECTRA={N_TIMING_SPECTRA}')
print(f'Batch sizes : {BATCH_SIZES}')
print(f'Fixed peak budget (CUDA Graph requirement): N_PEAKS={N_PEAKS}')
print(f'Compile mode: {COMPILE_MODE} | dynamo cache_size_limit={torch._dynamo.config.cache_size_limit}')
print(f'Profiler    : warmup={PROF_WARMUP} active={PROF_ACTIVE} (bs=1 only)')
os.makedirs('results', exist_ok=True)

Device : cuda | GPU: NVIDIA L4 | VRAM: 23.6 GB
PyTorch: 2.7.1+cu128 | CUDA: 12.8
N_SUBSET=6000 | N_TIMING_SPECTRA=5000
Batch sizes : [1, 8, 32, 128, 512]
Fixed peak budget (CUDA Graph requirement): N_PEAKS=150
Compile mode: reduce-overhead | dynamo cache_size_limit=32
Profiler    : warmup=20 active=50 (bs=1 only)


In [4]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — Download MGF via HTTP range request (no full zip)
# ═══════════════════════════════════════════════════════════════════
from remotezip import RemoteZip
import shutil
 
ZIP_URL  = 'https://zenodo.org/records/12587317/files/mgf_data.zip?download=1'
TARGET   = 'multi-enzyme-simple.test.mgf'
MGF_PATH = TARGET
 
if not (os.path.exists(MGF_PATH) and os.path.getsize(MGF_PATH) > 1e6):
    with RemoteZip(ZIP_URL) as zf:
        src = next(n for n in zf.namelist() if TARGET in n)
        zf.extract(src, '.')
    if src != MGF_PATH and os.path.exists(src):
        shutil.move(src, MGF_PATH)
        top = src.split('/')[0]
        if os.path.isdir(top): shutil.rmtree(top, ignore_errors=True)
 
print(f'{MGF_PATH}  ({os.path.getsize(MGF_PATH)/1e6:.1f} MB)')

multi-enzyme-simple.test.mgf  (300.9 MB)


In [5]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — EDA: parse MGF + plots
# ═══════════════════════════════════════════════════════════════════
import re as _re
def parse_mgf(path):
    records, spec, peaks, in_s = [], {}, [], False
    with open(path, 'r', errors='replace') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            if line.upper() == 'BEGIN IONS':
                spec, peaks, in_s = {}, [], True
            elif line.upper() == 'END IONS':
                if in_s:
                    records.append({'pepmass': spec.get('_pm', 0.0),
                                    'charge':  spec.get('_ch', 1),
                                    'n_peaks': len(peaks)})
                in_s = False
            elif in_s:
                if '=' in line:
                    k, _, v = line.partition('='); k = k.strip().upper()
                    if k == 'PEPMASS': spec['_pm'] = float(v.strip().split()[0])
                    elif k == 'CHARGE': spec['_ch'] = int(_re.sub(r'[^\d]', '', v.strip()) or '1')
                else:
                    p = line.split()
                    if p:
                        try: peaks.append(float(p[0]))
                        except ValueError: pass
    return pd.DataFrame(records)
 
eda = parse_mgf(MGF_PATH)
print(f'Spectra : {len(eda):,}')
print(f'Charge  : +{eda.charge.min()} to +{eda.charge.max()} | '
      f'+2: {int((eda.charge==2).sum()):,}  +3: {int((eda.charge==3).sum()):,}')
print(f'm/z     : {eda.pepmass.min():.1f} – {eda.pepmass.max():.1f}')
print(f'Peaks   : {eda.n_peaks.mean():.0f} avg  (min {eda.n_peaks.min()}, max {eda.n_peaks.max()})')
 
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('EDA — multi-enzyme-simple.test.mgf', fontweight='bold')
vc = eda.charge.value_counts().sort_index()
axes[0].bar(vc.index.astype(str), vc.values, color='steelblue', edgecolor='white')
axes[0].set(title='Charge distribution', xlabel='Charge', ylabel='Count')
for bar, v in zip(axes[0].patches, vc.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(vc.values)*0.01,
                 f'{v:,}', ha='center', fontsize=7)
axes[1].hist(eda.pepmass.clip(upper=3000), bins=60, color='darkorange', edgecolor='white')
axes[1].set(title='Precursor m/z (clip @3000)', xlabel='m/z', ylabel='Count')
axes[2].hist(eda.n_peaks.clip(upper=500), bins=60, color='seagreen', edgecolor='white')
axes[2].set(title='Peaks/spectrum (clip @500)', xlabel='Peaks', ylabel='Count')
plt.tight_layout()
plt.savefig('results/eda_plots.png', dpi=150, bbox_inches='tight'); plt.show()
print('Saved: results/eda_plots.png')

Spectra : 106,933
Charge  : +1 to +8 | +2: 40,614  +3: 39,146
m/z     : 301.2 – 1604.3
Peaks   : 123 avg  (min 6, max 950)
Saved: results/eda_plots.png


In [6]:
 
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — Loading Casanovo model via Python API
# Uses the exact same checkpoint-finding function the CLI uses.
# ModelRunner handles all Lightning version compatibility internally.
# ═══════════════════════════════════════════════════════════════════
import appdirs
from casanovo.casanovo import _get_model_weights
from casanovo.denovo.model_runner import ModelRunner
from casanovo.config import Config
 
cache_dir = Path(appdirs.user_cache_dir("casanovo", False, opinion=False))
print(f'Cache dir : {cache_dir}')
ckpt_path  = _get_model_weights(cache_dir)   # finds cached or downloads
print(f'Checkpoint: {ckpt_path}  ({os.path.getsize(str(ckpt_path))/1e6:.0f} MB)')
 
config = Config()
runner = ModelRunner(config=config, model_filename=str(ckpt_path))
runner.initialize_tokenizer()
runner.initialize_model(train=False)
model  = runner.model.to(DEVICE).eval()
 
n_params = sum(p.numel() for p in model.parameters())
print(f'\nModel class : {type(model).__name__}')
print(f'Parameters  : {n_params/1e6:.1f}M')
print(f'dim_model   : {model.encoder.latent_spectrum.shape[-1]}')
print(f'Enc layers  : {model.encoder.transformer_encoder.num_layers}')
try:
    print(f'Dec layers  : {model.decoder.transformer_decoder.num_layers}')
except AttributeError:
    pass
print(f'n_beams     : {model.n_beams} | max_peptide_len: {model.max_peptide_len}')
# ── Verify NAR patches are live on the loaded model ─────────────────
print('\n── NAR verification on loaded model ──')
import inspect as _insp
try:
    _v_embed = 'tgt_mask = torch.zeros' in _insp.getsource(model.decoder.__class__.embed)
    _v_fwd   = 'zero_tokens'            in _insp.getsource(model._forward_step)
except (OSError, TypeError):
    _v_embed = (model.decoder.__class__.embed    is _nar_embed)
    _v_fwd   = (model.__class__._forward_step    is _nar_forward_step)

print(f'  Decoder embed → full attention (NAR) : {"✓" if _v_embed else "✗  — re-run Cell 0!"}')
print(f'  _forward_step → zero tokens (NAR)    : {"✓" if _v_fwd   else "✗  — re-run Cell 0!"}')
print(f'  beam_search_decode present (unused)  : {hasattr(model, "beam_search_decode")}')
if not (_v_embed and _v_fwd):
    raise RuntimeError('NAR patches not active on loaded model. Run Cell 0 first.')
print(f'  max_peptide_len = {model.max_peptide_len}')

Checkpoint directory not set in ModelRunner, no checkpoint files will be saved.
Configured residue(s) not in model alphabet: [Acetyl]-, [Carbamyl]-, N[Deamidated], C[Carbamidomethyl], M[Oxidation], [+25.980265]-, [Ammonia-loss]-, Q[Deamidated]


Cache dir : /home/zeus/.cache/casanovo
Checkpoint: /home/zeus/.cache/casanovo/casanovo_v5_0_0_v5_0_0.ckpt  (575 MB)

Model class : Spec2Pep
Parameters  : 47.9M
dim_model   : 512
Enc layers  : 9
Dec layers  : 9
n_beams     : 1 | max_peptide_len: 100

── NAR verification on loaded model ──
  Decoder embed → full attention (NAR) : ✓
  _forward_step → zero tokens (NAR)    : ✓
  beam_search_decode present (unused)  : True
  max_peptide_len = 100


In [7]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4b (NEW) — Compile encoder + decoder with torch.compile (CUDA Graphs)
# Insert as a NEW cell immediately after Cell 4 (model loading).
#
# These objects are created ONCE here and reused everywhere below
# (Cells 7, 8, 9) — creating a new torch.compile() wrapper discards any
# previously cached compilation, so re-creating it per cell would force
# an expensive recompile every time.
# ═══════════════════════════════════════════════════════════════════════
compiled_encoder = torch.compile(model.encoder, mode=COMPILE_MODE)
compiled_decoder = torch.compile(model.decoder, mode=COMPILE_MODE)

print(f'compiled_encoder / compiled_decoder created  (mode={COMPILE_MODE})')
print('Note: the FIRST call at each new (module, shape) combination triggers')
print('a one-time TorchInductor compile + CUDA Graph capture (10-60+ seconds).')
print('This happens automatically inside the warm-up loop of Cell 7 / 8 / 9')
print('below and is excluded from all timed results.')

compiled_encoder / compiled_decoder created  (mode=reduce-overhead)
Note: the FIRST call at each new (module, shape) combination triggers
a one-time TorchInductor compile + CUDA Graph capture (10-60+ seconds).
This happens automatically inside the warm-up loop of Cell 7 / 8 / 9
below and is excluded from all timed results.


In [8]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5 — Build 6 000-spectrum MGF subset + Lance DataModule
# Auto-rebuilds the Lance if N_SUBSET or max_charge changed from the
# previous run (e.g. old subset had only 100 spectra).
# ═══════════════════════════════════════════════════════════════════════
from casanovo.denovo.dataloaders import DeNovoDataModule
import shutil as _shutil

SUBSET_MGF = 'subset_profile.mgf'
LANCE_DIR  = os.path.join(os.getcwd(), 'lance_cache')
os.makedirs(LANCE_DIR, exist_ok=True)

MODEL_MAX_CHARGE = model.decoder.charge_encoder.num_embeddings
print(f'Model max_charge: {MODEL_MAX_CHARGE}')

# ── Helpers ──────────────────────────────────────────────────────────
def _count_mgf_spectra(path):
    if not os.path.exists(path):
        return 0
    with open(path, 'r', errors='replace') as f:
        return f.read().count('BEGIN IONS')

def write_subset_mgf(src, dest, n):
    count, buf, in_s = 0, [], False
    with open(src, 'r', errors='replace') as fin, open(dest, 'w') as fout:
        for line in fin:
            if count >= n:
                break
            if line.strip().upper() == 'BEGIN IONS':
                in_s = True; buf = [line]
            elif line.strip().upper() == 'END IONS':
                buf.append(line); fout.writelines(buf)
                count += 1; in_s = False; buf = []
            elif in_s:
                buf.append(line)
    return count

# ── Rebuild MGF subset if it is missing or too small ────────────────
_existing_n = _count_mgf_spectra(SUBSET_MGF)
if _existing_n < N_SUBSET:
    if _existing_n > 0:
        print(f'Old SUBSET_MGF has only {_existing_n} spectra (need {N_SUBSET}). Rebuilding…')
        os.remove(SUBSET_MGF)
    wrote = write_subset_mgf(MGF_PATH, SUBSET_MGF, N_SUBSET)
    print(f'Created: {SUBSET_MGF}  ({wrote} spectra)')
else:
    print(f'Reusing: {SUBSET_MGF}  ({_existing_n} spectra)')

# ── Rebuild Lance if N_SUBSET or max_charge changed ──────────────────
# Cache key encodes both so any change triggers a clean rebuild.
_mc_marker  = os.path.join(LANCE_DIR, '.cache_key')
_lance_test = os.path.join(LANCE_DIR, 'test.lance')
_cache_key  = f'{MODEL_MAX_CHARGE}_{N_SUBSET}'
_prev_key   = open(_mc_marker).read().strip() if os.path.exists(_mc_marker) else 'none'

if _prev_key != _cache_key:
    if os.path.exists(_lance_test):
        _shutil.rmtree(_lance_test)
        print(f'Deleted stale Lance (was {_prev_key!r}, now {_cache_key!r})')
    with open(_mc_marker, 'w') as f:
        f.write(_cache_key)
    print(f'Building Lance  cache_key={_cache_key} …')
else:
    print(f'Reusing Lance   cache_key={_cache_key} ✓')

# ── DataModule (bs=1 for setup / sanity check) ───────────────────────
dm = DeNovoDataModule(
    lance_dir=LANCE_DIR,
    test_paths=[SUBSET_MGF],
    eval_batch_size=1,
    tokenizer=runner.model.tokenizer,
    max_charge=MODEL_MAX_CHARGE,
    n_workers=0,
)
dm.setup(stage='test', annotated=False)
print('DataModule (bs=1) ready.')

# ── Sanity-check first batch ─────────────────────────────────────────
_it   = iter(dm.predict_dataloader())
_b    = next(_it); del _it
_mzs, _ints, _precs, _ = model._process_batch(_b)
_charge = _precs[0, 1].item() if _precs.ndim == 2 else _precs[1].item()
assert _charge <= MODEL_MAX_CHARGE, \
    f'Charge {_charge} > model max_charge {MODEL_MAX_CHARGE}'
print(f'First batch  mzs={_mzs.shape}  precs={_precs.shape}')
print(f'Precursor [0]: mass={_precs[0,0]:.1f}  charge={_charge:.0f}  mz={_precs[0,2]:.1f} ✓')
print(f'Subset ready: {N_SUBSET} spectra  (timing target: {N_TIMING_SPECTRA})')

Model max_charge: 4
Reusing: subset_profile.mgf  (6000 spectra)
Reusing Lance   cache_key=4_6000 ✓


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

DataModule (bs=1) ready.
First batch  mzs=torch.Size([1, 42])  precs=torch.Size([1, 3])
Precursor [0]: mass=3370.5  charge=3  mz=1124.5 ✓
Subset ready: 6000 spectra  (timing target: 5000)


In [9]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 — Baseline NAR Timing (eager mode, fixed N_PEAKS padding)
# Un-optimized NAR pipeline (PR #548 patch only) — the baseline for
# Cell 7's torch.compile + CUDA Graphs comparison.
# ═══════════════════════════════════════════════════════════════════════
import threading

_gpu_samples_bl = []
_stop_gpu_bl    = threading.Event()

def _gpu_monitor_bl():
    import subprocess as _sp
    while not _stop_gpu_bl.is_set():
        r = _sp.run(['nvidia-smi',
                     '--query-gpu=utilization.gpu,memory.used',
                     '--format=csv,noheader,nounits'],
                    capture_output=True, text=True)
        if r.returncode == 0:
            try:
                u, m = r.stdout.strip().split(', ')
                _gpu_samples_bl.append((int(u), float(m) / 1024))
            except Exception:
                pass
        time.sleep(0.5)

threading.Thread(target=_gpu_monitor_bl, daemon=True).start()

timing_baseline = {}

for bs in BATCH_SIZES:
    print(f'\n══ Timing BASELINE NAR (eager)  batch_size={bs:4d} ══')

    _dm_bs = DeNovoDataModule(
        lance_dir=LANCE_DIR,
        test_paths=[SUBSET_MGF],
        eval_batch_size=bs,
        tokenizer=runner.model.tokenizer,
        max_charge=MODEL_MAX_CHARGE,
        n_workers=0,
    )
    _dm_bs.setup(stage='test', annotated=False)

    # ── Warm-up ───────────────────────────────────────────────────────
    _w_iter = iter(_dm_bs.predict_dataloader())
    with torch.no_grad():
        for _w in range(N_WARMUP_BATCHES):
            try:
                _wb = next(_w_iter)
            except StopIteration:
                break
            _wm, _wi, _wp, _ = model._process_batch(_wb)
            _wm = _wm.to(DEVICE); _wi = _wi.to(DEVICE); _wp = _wp.to(DEVICE)
            _wm, _wi, _ = pad_or_select_to_fixed_peaks(_wm, _wi)
            _wme, _wmk = model.encoder(_wm, _wi)
            _wz = torch.zeros((_wm.shape[0], model.max_peptide_len),
                               dtype=torch.long, device=DEVICE)
            model.decoder(tokens=_wz, memory=_wme,
                          memory_key_padding_mask=_wmk, precursors=_wp)
    del _w_iter
    _sync()

    # ── Timing storage ───────────────────────────────────────────────
    _t = {k: [] for k in ['fetch', 'h2d', 'enc', 'nar', 'write', 'total', 'tp']}
    _n_truncated = 0
    _loader = _dm_bs.predict_dataloader()
    _it     = iter(_loader)
    n_spec  = 0
    pbar    = tqdm(total=N_TIMING_SPECTRA, desc=f'  bs={bs}', unit='spec')

    while n_spec < N_TIMING_SPECTRA:
        _sync(); t0 = time.perf_counter()
        try:
            batch = next(_it)
        except StopIteration:
            _it = iter(_loader)
            batch = next(_it)
        t_fetch = (time.perf_counter() - t0) * 1000

        _sync(); t0 = time.perf_counter()
        mzs, ints, precs, _ = model._process_batch(batch)
        mzs   = mzs.to(DEVICE)
        ints  = ints.to(DEVICE)
        precs = precs.to(DEVICE)
        mzs, ints, _ntrunc = pad_or_select_to_fixed_peaks(mzs, ints)
        _n_truncated += _ntrunc
        _sync(); t_h2d = (time.perf_counter() - t0) * 1000
        actual_bs = mzs.shape[0]

        with torch.no_grad():
            _sync(); t0 = time.perf_counter()
            memories, mem_masks = model.encoder(mzs, ints)
            _sync(); t_enc = (time.perf_counter() - t0) * 1000

            zero_tokens = torch.zeros(
                (actual_bs, model.max_peptide_len),
                dtype=torch.long, device=DEVICE)
            _sync(); t0 = time.perf_counter()
            scores = model.decoder(
                tokens=zero_tokens,
                memory=memories,
                memory_key_padding_mask=mem_masks,
                precursors=precs,
            )
            _sync(); t_nar = (time.perf_counter() - t0) * 1000

        t0 = time.perf_counter()
        pred_tok = scores.argmax(dim=-1).cpu()
        _out = [{'tokens': tok.tolist()} for tok in pred_tok]
        t_write = (time.perf_counter() - t0) * 1000

        t_total = t_fetch + t_h2d + t_enc + t_nar + t_write

        _t['fetch'].append(t_fetch  / actual_bs)
        _t['h2d'].append(t_h2d     / actual_bs)
        _t['enc'].append(t_enc     / actual_bs)
        _t['nar'].append(t_nar     / actual_bs)
        _t['write'].append(t_write / actual_bs)
        _t['total'].append(t_total / actual_bs)
        _t['tp'].append(actual_bs  / (t_total / 1000))

        n_spec += actual_bs
        pbar.update(actual_bs)
        if n_spec >= N_TIMING_SPECTRA:
            break

    pbar.close()

    def _p(a, q): return np.percentile(a, q)
    timing_baseline[bs] = {
        'n_spec'     : n_spec,
        'n_batches'  : len(_t['total']),
        'n_truncated': _n_truncated,
        'fetch_mean' : np.mean(_t['fetch']),
        'h2d_mean'   : np.mean(_t['h2d']),
        'enc_mean'   : np.mean(_t['enc']),
        'nar_mean'   : np.mean(_t['nar']),
        'write_mean' : np.mean(_t['write']),
        'total_mean' : np.mean(_t['total']),
        'total_p50'  : _p(_t['total'], 50),
        'total_p95'  : _p(_t['total'], 95),
        'throughput' : np.mean(_t['tp']),
        'raw'        : _t,
    }
    s = timing_baseline[bs]
    print(f'  spectra={n_spec}  batches={len(_t["total"])}  truncated={_n_truncated}')
    print(f'  total : {s["total_mean"]:7.2f} ms/spec  p50={s["total_p50"]:.2f}  p95={s["total_p95"]:.2f}')
    print(f'  enc   : {s["enc_mean"]:7.2f} ms/spec  nar={s["nar_mean"]:.2f} ms/spec')
    print(f'  tp    : {s["throughput"]:.1f} spec/s')

    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

_stop_gpu_bl.set()
time.sleep(1.0)

gpu_util_mean_bl = np.mean([s[0] for s in _gpu_samples_bl]) if _gpu_samples_bl else 0
gpu_vram_peak_bl = np.max([s[1] for s in _gpu_samples_bl]) if _gpu_samples_bl else 0

b1 = timing_baseline[1]
rb = b1['raw']
df_stage_baseline = pd.DataFrame([
    {'Stage': 'DataLoader fetch',     'mean_ms': b1['fetch_mean'],
     'p50_ms': np.percentile(rb['fetch'], 50), 'p95_ms': np.percentile(rb['fetch'], 95)},
    {'Stage': 'H2D + fixed-peak pad', 'mean_ms': b1['h2d_mean'],
     'p50_ms': np.percentile(rb['h2d'],   50), 'p95_ms': np.percentile(rb['h2d'],   95)},
    {'Stage': 'SpectrumEncoder',      'mean_ms': b1['enc_mean'],
     'p50_ms': np.percentile(rb['enc'],   50), 'p95_ms': np.percentile(rb['enc'],   95)},
    {'Stage': 'NAR Decoder (1 pass)', 'mean_ms': b1['nar_mean'],
     'p50_ms': np.percentile(rb['nar'],   50), 'p95_ms': np.percentile(rb['nar'],   95)},
    {'Stage': 'Output write',         'mean_ms': b1['write_mean'],
     'p50_ms': np.percentile(rb['write'], 50), 'p95_ms': np.percentile(rb['write'], 95)},
    {'Stage': 'TOTAL per spectrum',   'mean_ms': b1['total_mean'],
     'p50_ms': b1['total_p50'],                'p95_ms': b1['total_p95']},
]).round(3)

df_throughput_baseline = pd.DataFrame([{
    'batch_size'        : bs,
    'total_ms_per_spec' : timing_baseline[bs]['total_mean'],
    'throughput_spec_s' : timing_baseline[bs]['throughput'],
    'enc_ms_per_spec'   : timing_baseline[bs]['enc_mean'],
    'nar_ms_per_spec'   : timing_baseline[bs]['nar_mean'],
    'p50_ms'            : timing_baseline[bs]['total_p50'],
    'p95_ms'            : timing_baseline[bs]['total_p95'],
} for bs in BATCH_SIZES]).round(3)

print(f'\n── Stage breakdown (BASELINE NAR, bs=1, {b1["n_spec"]} spectra) ──')
print(df_stage_baseline.to_string(index=False))
print(f'\n── Multi-batch throughput (BASELINE NAR) ──')
print(df_throughput_baseline.to_string(index=False))
print(f'\nGPU util (mean): {gpu_util_mean_bl:.0f}%  |  Peak VRAM: {gpu_vram_peak_bl:.2f} GB')

_total_ms_bl = b1['total_mean']
_msg_35 = 'MEETS ✓' if _total_ms_bl <= 35 else f'FAILS — {_total_ms_bl:.1f} ms'
_msg_50 = 'MEETS ✓' if _total_ms_bl <= 50 else f'FAILS — {_total_ms_bl:.1f} ms'
_msg_10 = 'MEETS ✓' if _total_ms_bl <= 10 else f'FAILS — {_total_ms_bl:.1f} ms'
print(f'35 ms (~29 Hz) target (bs=1): {_msg_35}')
print(f'50 ms (20 Hz)  target (bs=1): {_msg_50}')
print(f'10 ms (100 Hz) target (bs=1): {_msg_10}')

df_stage_baseline.to_csv('results/nar_baseline_stage_timing_bs1.csv', index=False)
df_throughput_baseline.to_csv('results/nar_baseline_throughput_all_bs.csv', index=False)
print('\nSaved: nar_baseline_stage_timing_bs1.csv | nar_baseline_throughput_all_bs.csv')


══ Timing BASELINE NAR (eager)  batch_size=   1 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=1: 100%|██████████| 5000/5000 [01:57<00:00, 42.66spec/s]


  spectra=5000  batches=5000  truncated=0
  total :   23.04 ms/spec  p50=22.06  p95=30.57
  enc   :    8.12 ms/spec  nar=13.02 ms/spec
  tp    : 43.9 spec/s

══ Timing BASELINE NAR (eager)  batch_size=   8 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

...

  bs=128: 5120spec [00:09, 519.68spec/s]                        

  spectra=5120  batches=40  truncated=0
  total :    1.91 ms/spec  p50=1.90  p95=2.01
  enc   :    0.54 ms/spec  nar=1.16 ms/spec
  tp    : 523.4 spec/s

══ Timing BASELINE NAR (eager)  batch_size= 512 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=512: 5120spec [00:10, 489.86spec/s]                        


  spectra=5120  batches=10  truncated=0
  total :    2.04 ms/spec  p50=2.04  p95=2.07
  enc   :    0.62 ms/spec  nar=1.22 ms/spec
  tp    : 490.7 spec/s

── Stage breakdown (BASELINE NAR, bs=1, 5000 spectra) ──
               Stage  mean_ms  p50_ms  p95_ms
    DataLoader fetch    1.488   1.440   1.960
H2D + fixed-peak pad    0.295   0.291   0.423
     SpectrumEncoder    8.124   7.679  11.612
NAR Decoder (1 pass)   13.020  12.474  16.659
        Output write    0.110   0.103   0.143
  TOTAL per spectrum   23.036  22.062  30.568

── Multi-batch throughput (BASELINE NAR) ──
 batch_size  total_ms_per_spec  throughput_spec_s  enc_ms_per_spec  nar_ms_per_spec  p50_ms  p95_ms
          1             23.036             43.928            8.124           13.020  22.062  30.568
          8              3.259            311.850            1.083            1.744   3.068   4.366
         32              1.876            533.932            0.570            1.020   1.866   2.017
        128           

In [10]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 7 — Optimized NAR Timing (torch.compile + CUDA Graphs)
# FIX: torch.compiler.cudagraph_mark_step_begin() called once per
# iteration, right before compiled_encoder(...), because we chain two
# SEPARATELY compiled functions (encoder -> decoder). Without this, the
# CUDA Graph Trees allocator can't reliably tell where one "step" ends
# and the next begins, and overwrites the encoder's output buffer before
# the decoder finishes reading it.
# Docs: https://docs.pytorch.org/docs/stable/torch.compiler_cudagraph_trees.html
# ═══════════════════════════════════════════════════════════════════════
_gpu_samples_cp = []
_stop_gpu_cp    = threading.Event()

def _gpu_monitor_cp():
    import subprocess as _sp
    while not _stop_gpu_cp.is_set():
        r = _sp.run(['nvidia-smi',
                     '--query-gpu=utilization.gpu,memory.used',
                     '--format=csv,noheader,nounits'],
                    capture_output=True, text=True)
        if r.returncode == 0:
            try:
                u, m = r.stdout.strip().split(', ')
                _gpu_samples_cp.append((int(u), float(m) / 1024))
            except Exception:
                pass
        time.sleep(0.5)

threading.Thread(target=_gpu_monitor_cp, daemon=True).start()

def _mark_step():
    if DEVICE == 'cuda':
        torch.compiler.cudagraph_mark_step_begin()

timing_compiled = {}

for bs in BATCH_SIZES:
    print(f'\n══ Timing COMPILED NAR (torch.compile+CUDA Graphs)  batch_size={bs:4d} ══')

    _dm_bs = DeNovoDataModule(
        lance_dir=LANCE_DIR,
        test_paths=[SUBSET_MGF],
        eval_batch_size=bs,
        tokenizer=runner.model.tokenizer,
        max_charge=MODEL_MAX_CHARGE,
        n_workers=0,
    )
    _dm_bs.setup(stage='test', annotated=False)

    print(f'  Compiling for bs={bs} (first call only — may take 10-60s)…')
    _compile_t0 = time.perf_counter()
    _w_iter = iter(_dm_bs.predict_dataloader())
    with torch.no_grad():
        for _w in range(N_WARMUP_BATCHES):
            try:
                _wb = next(_w_iter)
            except StopIteration:
                break
            _wm, _wi, _wp, _ = model._process_batch(_wb)
            _wm = _wm.to(DEVICE); _wi = _wi.to(DEVICE); _wp = _wp.to(DEVICE)
            _wm, _wi, _ = pad_or_select_to_fixed_peaks(_wm, _wi)
            _mark_step()
            _wme, _wmk = compiled_encoder(_wm, _wi)
            _wz = torch.zeros((_wm.shape[0], model.max_peptide_len),
                               dtype=torch.long, device=DEVICE)
            compiled_decoder(tokens=_wz, memory=_wme,
                             memory_key_padding_mask=_wmk, precursors=_wp)
    del _w_iter
    _sync()
    _compile_elapsed = time.perf_counter() - _compile_t0
    print(f'  Warm-up + compile done in {_compile_elapsed:.1f}s (one-time cost, excluded below)')

    _t = {k: [] for k in ['fetch', 'h2d', 'enc', 'nar', 'write', 'total', 'tp']}
    _n_truncated = 0
    _loader = _dm_bs.predict_dataloader()
    _it     = iter(_loader)
    n_spec  = 0
    pbar    = tqdm(total=N_TIMING_SPECTRA, desc=f'  bs={bs}', unit='spec')

    while n_spec < N_TIMING_SPECTRA:
        _sync(); t0 = time.perf_counter()
        try:
            batch = next(_it)
        except StopIteration:
            _it = iter(_loader)
            batch = next(_it)
        t_fetch = (time.perf_counter() - t0) * 1000

        _sync(); t0 = time.perf_counter()
        mzs, ints, precs, _ = model._process_batch(batch)
        mzs   = mzs.to(DEVICE)
        ints  = ints.to(DEVICE)
        precs = precs.to(DEVICE)
        mzs, ints, _ntrunc = pad_or_select_to_fixed_peaks(mzs, ints)
        _n_truncated += _ntrunc
        _sync(); t_h2d = (time.perf_counter() - t0) * 1000
        actual_bs = mzs.shape[0]

        with torch.no_grad():
            _mark_step()   # ← marks the start of a new CUDA-Graph "step"
                           #   before the encoder->decoder chain below
            _sync(); t0 = time.perf_counter()
            memories, mem_masks = compiled_encoder(mzs, ints)
            _sync(); t_enc = (time.perf_counter() - t0) * 1000

            zero_tokens = torch.zeros(
                (actual_bs, model.max_peptide_len),
                dtype=torch.long, device=DEVICE)
            _sync(); t0 = time.perf_counter()
            scores = compiled_decoder(
                tokens=zero_tokens,
                memory=memories,
                memory_key_padding_mask=mem_masks,
                precursors=precs,
            )
            _sync(); t_nar = (time.perf_counter() - t0) * 1000

        t0 = time.perf_counter()
        pred_tok = scores.argmax(dim=-1).cpu()
        _out = [{'tokens': tok.tolist()} for tok in pred_tok]
        t_write = (time.perf_counter() - t0) * 1000

        t_total = t_fetch + t_h2d + t_enc + t_nar + t_write

        _t['fetch'].append(t_fetch  / actual_bs)
        _t['h2d'].append(t_h2d     / actual_bs)
        _t['enc'].append(t_enc     / actual_bs)
        _t['nar'].append(t_nar     / actual_bs)
        _t['write'].append(t_write / actual_bs)
        _t['total'].append(t_total / actual_bs)
        _t['tp'].append(actual_bs  / (t_total / 1000))

        n_spec += actual_bs
        pbar.update(actual_bs)
        if n_spec >= N_TIMING_SPECTRA:
            break

    pbar.close()

    def _p(a, q): return np.percentile(a, q)
    timing_compiled[bs] = {
        'n_spec'     : n_spec,
        'n_batches'  : len(_t['total']),
        'n_truncated': _n_truncated,
        'compile_s'  : _compile_elapsed,
        'fetch_mean' : np.mean(_t['fetch']),
        'h2d_mean'   : np.mean(_t['h2d']),
        'enc_mean'   : np.mean(_t['enc']),
        'nar_mean'   : np.mean(_t['nar']),
        'write_mean' : np.mean(_t['write']),
        'total_mean' : np.mean(_t['total']),
        'total_p50'  : _p(_t['total'], 50),
        'total_p95'  : _p(_t['total'], 95),
        'throughput' : np.mean(_t['tp']),
        'raw'        : _t,
    }
    s = timing_compiled[bs]
    print(f'  spectra={n_spec}  batches={len(_t["total"])}  truncated={_n_truncated}')
    print(f'  total : {s["total_mean"]:7.2f} ms/spec  p50={s["total_p50"]:.2f}  p95={s["total_p95"]:.2f}')
    print(f'  enc   : {s["enc_mean"]:7.2f} ms/spec  nar={s["nar_mean"]:.2f} ms/spec')
    print(f'  tp    : {s["throughput"]:.1f} spec/s')

    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

_stop_gpu_cp.set()
time.sleep(1.0)

gpu_util_mean_cp = np.mean([s[0] for s in _gpu_samples_cp]) if _gpu_samples_cp else 0
gpu_vram_peak_cp = np.max([s[1] for s in _gpu_samples_cp]) if _gpu_samples_cp else 0

c1 = timing_compiled[1]
rc = c1['raw']
df_stage_compiled = pd.DataFrame([
    {'Stage': 'DataLoader fetch',          'mean_ms': c1['fetch_mean'],
     'p50_ms': np.percentile(rc['fetch'], 50), 'p95_ms': np.percentile(rc['fetch'], 95)},
    {'Stage': 'H2D + fixed-peak pad',      'mean_ms': c1['h2d_mean'],
     'p50_ms': np.percentile(rc['h2d'],   50), 'p95_ms': np.percentile(rc['h2d'],   95)},
    {'Stage': 'SpectrumEncoder (compiled)','mean_ms': c1['enc_mean'],
     'p50_ms': np.percentile(rc['enc'],   50), 'p95_ms': np.percentile(rc['enc'],   95)},
    {'Stage': 'NAR Decoder (compiled, 1 pass)', 'mean_ms': c1['nar_mean'],
     'p50_ms': np.percentile(rc['nar'],   50), 'p95_ms': np.percentile(rc['nar'],   95)},
    {'Stage': 'Output write',              'mean_ms': c1['write_mean'],
     'p50_ms': np.percentile(rc['write'], 50), 'p95_ms': np.percentile(rc['write'], 95)},
    {'Stage': 'TOTAL per spectrum',        'mean_ms': c1['total_mean'],
     'p50_ms': c1['total_p50'],                'p95_ms': c1['total_p95']},
]).round(3)

df_throughput_compiled = pd.DataFrame([{
    'batch_size'        : bs,
    'total_ms_per_spec' : timing_compiled[bs]['total_mean'],
    'throughput_spec_s' : timing_compiled[bs]['throughput'],
    'enc_ms_per_spec'   : timing_compiled[bs]['enc_mean'],
    'nar_ms_per_spec'   : timing_compiled[bs]['nar_mean'],
    'p50_ms'            : timing_compiled[bs]['total_p50'],
    'p95_ms'            : timing_compiled[bs]['total_p95'],
    'compile_s'         : timing_compiled[bs]['compile_s'],
} for bs in BATCH_SIZES]).round(3)

print(f'\n── Stage breakdown (COMPILED NAR, bs=1, {c1["n_spec"]} spectra) ──')
print(df_stage_compiled.to_string(index=False))
print(f'\n── Multi-batch throughput (COMPILED NAR) ──')
print(df_throughput_compiled.to_string(index=False))
print(f'\nGPU util (mean): {gpu_util_mean_cp:.0f}%  |  Peak VRAM: {gpu_vram_peak_cp:.2f} GB')

_total_ms_cp = c1['total_mean']
_msg_35c = 'MEETS ✓' if _total_ms_cp <= 35 else f'FAILS — {_total_ms_cp:.1f} ms'
_msg_50c = 'MEETS ✓' if _total_ms_cp <= 50 else f'FAILS — {_total_ms_cp:.1f} ms'
_msg_10c = 'MEETS ✓' if _total_ms_cp <= 10 else f'FAILS — {_total_ms_cp:.1f} ms'
print(f'35 ms (~29 Hz) target (bs=1): {_msg_35c}')
print(f'50 ms (20 Hz)  target (bs=1): {_msg_50c}')
print(f'10 ms (100 Hz) target (bs=1): {_msg_10c}')

_speedup_bs1 = timing_baseline[1]['total_mean'] / max(c1['total_mean'], 0.001)
print(f'\nSpeedup vs baseline NAR (bs=1): {_speedup_bs1:.2f}×')

df_stage_compiled.to_csv('results/nar_compiled_stage_timing_bs1.csv', index=False)
df_throughput_compiled.to_csv('results/nar_compiled_throughput_all_bs.csv', index=False)
print('\nSaved: nar_compiled_stage_timing_bs1.csv | nar_compiled_throughput_all_bs.csv')


══ Timing COMPILED NAR (torch.compile+CUDA Graphs)  batch_size=   1 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  Compiling for bs=1 (first call only — may take 10-60s)…
  Warm-up + compile done in 7.2s (one-time cost, excluded below)


  bs=1: 100%|██████████| 5000/5000 [01:59<00:00, 41.74spec/s]


  spectra=5000  batches=5000  truncated=0
  total :   23.48 ms/spec  p50=22.54  p95=30.18
  enc   :    8.07 ms/spec  nar=13.40 ms/spec
  tp    : 43.1 spec/s

══ Timing COMPILED NAR (torch.compile+CUDA Graphs)  batch_size=   8 ══


...

  spectra=5120  batches=40  truncated=0
  total :    1.93 ms/spec  p50=1.91  p95=2.03
  enc   :    0.53 ms/spec  nar=1.18 ms/spec
  tp    : 517.7 spec/s

══ Timing COMPILED NAR (torch.compile+CUDA Graphs)  batch_size= 512 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  Compiling for bs=512 (first call only — may take 10-60s)…
  Warm-up + compile done in 17.1s (one-time cost, excluded below)


  bs=512: 5120spec [00:10, 484.38spec/s]                        


  spectra=5120  batches=10  truncated=0
  total :    2.06 ms/spec  p50=2.06  p95=2.11
  enc   :    0.62 ms/spec  nar=1.24 ms/spec
  tp    : 485.3 spec/s

── Stage breakdown (COMPILED NAR, bs=1, 5000 spectra) ──
                         Stage  mean_ms  p50_ms  p95_ms
              DataLoader fetch    1.569   1.520   2.071
          H2D + fixed-peak pad    0.318   0.315   0.457
    SpectrumEncoder (compiled)    8.067   7.657  11.264
NAR Decoder (compiled, 1 pass)   13.399  12.867  16.952
                  Output write    0.126   0.120   0.160
            TOTAL per spectrum   23.479  22.535  30.176

── Multi-batch throughput (COMPILED NAR) ──
 batch_size  total_ms_per_spec  throughput_spec_s  enc_ms_per_spec  nar_ms_per_spec  p50_ms  p95_ms  compile_s
          1             23.479             43.080            8.067           13.399  22.535  30.176      7.206
          8              3.325            303.543            1.050            1.814   3.221   4.211      7.746
         32        

In [11]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 8 — torch.profiler: Baseline NAR vs Compiled NAR (bs=1, ≥50 spectra)
# FIX: same cudagraph_mark_step_begin() fix as Cell 7, applied wherever
# compiled_encoder's output feeds into compiled_decoder.
# ═══════════════════════════════════════════════════════════════════════
ACTS           = ([ProfilerActivity.CPU, ProfilerActivity.CUDA]
                  if DEVICE == 'cuda' else [ProfilerActivity.CPU])
SORT_KEY       = 'cpu_time_total'
N_PROF_BATCHES = PROF_WARMUP + PROF_ACTIVE

print(f'Pre-fetching {N_PROF_BATCHES} bs=1 batches for profiler…')
_dm_prof = DeNovoDataModule(
    lance_dir=LANCE_DIR,
    test_paths=[SUBSET_MGF],
    eval_batch_size=1,
    tokenizer=runner.model.tokenizer,
    max_charge=MODEL_MAX_CHARGE,
    n_workers=0,
)
_dm_prof.setup(stage='test', annotated=False)

_prof_batches, _skipped, _ntrunc_prof = [], 0, 0
for _b in _dm_prof.predict_dataloader():
    _mz, _it, _pr, _ = model._process_batch(_b)
    if _pr[0, 1].item() > MODEL_MAX_CHARGE:
        _skipped += 1; continue
    _mz = _mz.to(DEVICE); _it = _it.to(DEVICE); _pr = _pr.to(DEVICE)
    _mz, _it, _nt = pad_or_select_to_fixed_peaks(_mz, _it)
    _ntrunc_prof += _nt
    _prof_batches.append((_mz, _it, _pr))
    if len(_prof_batches) >= N_PROF_BATCHES:
        break

if _skipped:
    print(f'  Skipped {_skipped} out-of-range-charge spectra')
if len(_prof_batches) == 0:
    raise RuntimeError('No valid batches for profiling.')
while len(_prof_batches) < N_PROF_BATCHES:
    _prof_batches.extend(_prof_batches[:N_PROF_BATCHES - len(_prof_batches)])
print(f'Using {len(_prof_batches)} batches  (truncated to N_PEAKS: {_ntrunc_prof})')

_zero_toks = torch.zeros((1, model.max_peptide_len), dtype=torch.long, device=DEVICE)

def _mark_step():
    if DEVICE == 'cuda':
        torch.compiler.cudagraph_mark_step_begin()

def _run_profiler(encoder_fn, decoder_fn, label, trace_path, txt_path, n_warm=10, use_graphs=False):
    with torch.no_grad():
        for _mz, _it, _pr in _prof_batches[:n_warm]:
            if use_graphs:
                _mark_step()
            _me, _mk = encoder_fn(_mz, _it)
            decoder_fn(tokens=_zero_toks, memory=_me,
                      memory_key_padding_mask=_mk, precursors=_pr)
    _sync()

    _store = {}
    def _on_ready(p):
        p.export_chrome_trace(trace_path)
        _store['tbl']  = p.key_averages().table(sort_by=SORT_KEY, row_limit=12)
        _store['avgs'] = p.key_averages()

    with profile(
        activities=ACTS,
        record_shapes=True,
        schedule=schedule(wait=0, warmup=PROF_WARMUP, active=PROF_ACTIVE),
        on_trace_ready=_on_ready,
    ) as p:
        with torch.no_grad():
            for _mz, _it, _pr in _prof_batches:
                if use_graphs:
                    _mark_step()
                with record_function(label):
                    _me, _mk = encoder_fn(_mz, _it)
                    decoder_fn(tokens=_zero_toks, memory=_me,
                              memory_key_padding_mask=_mk, precursors=_pr)
                _sync()
                p.step()

    print(_store.get('tbl', '(no profiler data)'))
    with open(txt_path, 'w') as fh:
        fh.write(f'{label}  bs=1  warmup={PROF_WARMUP}  active={PROF_ACTIVE}\n')
        fh.write('=' * 64 + '\n')
        fh.write(str(_store.get('tbl', 'no data')))
    print(f'Chrome trace → {trace_path}')
    if DEVICE == 'cuda':
        torch.cuda.synchronize(); torch.cuda.empty_cache()
    return _store

# ════════════════════════════════════════════════════════════════════
# A) BASELINE — SpectrumEncoder only
# ════════════════════════════════════════════════════════════════════
print('\n── A) torch.profiler: BASELINE SpectrumEncoder (eager, bs=1) ──')
_store_bl_enc = {}
def _on_ready_bl_enc(p):
    p.export_chrome_trace('results/trace_baseline_encoder.json')
    _store_bl_enc['tbl']  = p.key_averages().table(sort_by=SORT_KEY, row_limit=10)
    _store_bl_enc['avgs'] = p.key_averages()
with torch.no_grad():
    for _mz, _it, _pr in _prof_batches[:10]:
        model.encoder(_mz, _it)
_sync()
with profile(activities=ACTS, record_shapes=True,
             schedule=schedule(wait=0, warmup=PROF_WARMUP, active=PROF_ACTIVE),
             on_trace_ready=_on_ready_bl_enc) as p_bl_enc:
    with torch.no_grad():
        for _mz, _it, _pr in _prof_batches:
            with record_function('baseline_encoder'):
                model.encoder(_mz, _it)
            _sync(); p_bl_enc.step()
print(_store_bl_enc.get('tbl', '(no data)'))
with open('results/profiler_baseline_encoder.txt', 'w') as fh:
    fh.write(f'BASELINE SpectrumEncoder  bs=1  warmup={PROF_WARMUP}  active={PROF_ACTIVE}\n')
    fh.write('=' * 64 + '\n')
    fh.write(str(_store_bl_enc.get('tbl', 'no data')))
print('Chrome trace → results/trace_baseline_encoder.json')
if DEVICE == 'cuda': torch.cuda.synchronize(); torch.cuda.empty_cache()

# ════════════════════════════════════════════════════════════════════
# B) BASELINE — Full NAR forward
# ════════════════════════════════════════════════════════════════════
print('\n── B) torch.profiler: BASELINE Full NAR forward (eager, bs=1) ──')
_store_bl_full = _run_profiler(
    model.encoder, model.decoder, 'baseline_full_forward',
    'results/trace_baseline_full.json', 'results/profiler_baseline_full.txt',
    use_graphs=False)

# ════════════════════════════════════════════════════════════════════
# C) COMPILED — SpectrumEncoder only
# ════════════════════════════════════════════════════════════════════
print('\n── C) torch.profiler: COMPILED SpectrumEncoder (torch.compile+CUDA Graphs, bs=1) ──')
_store_cp_enc = {}
def _on_ready_cp_enc(p):
    p.export_chrome_trace('results/trace_compiled_encoder.json')
    _store_cp_enc['tbl']  = p.key_averages().table(sort_by=SORT_KEY, row_limit=10)
    _store_cp_enc['avgs'] = p.key_averages()
with torch.no_grad():
    for _mz, _it, _pr in _prof_batches[:10]:
        _mark_step()
        compiled_encoder(_mz, _it)
_sync()
with profile(activities=ACTS, record_shapes=True,
             schedule=schedule(wait=0, warmup=PROF_WARMUP, active=PROF_ACTIVE),
             on_trace_ready=_on_ready_cp_enc) as p_cp_enc:
    with torch.no_grad():
        for _mz, _it, _pr in _prof_batches:
            _mark_step()
            with record_function('compiled_encoder'):
                compiled_encoder(_mz, _it)
            _sync(); p_cp_enc.step()
print(_store_cp_enc.get('tbl', '(no data)'))
with open('results/profiler_compiled_encoder.txt', 'w') as fh:
    fh.write(f'COMPILED SpectrumEncoder  bs=1  warmup={PROF_WARMUP}  active={PROF_ACTIVE}\n')
    fh.write('=' * 64 + '\n')
    fh.write(str(_store_cp_enc.get('tbl', 'no data')))
print('Chrome trace → results/trace_compiled_encoder.json')
if DEVICE == 'cuda': torch.cuda.synchronize(); torch.cuda.empty_cache()

# ════════════════════════════════════════════════════════════════════
# D) COMPILED — Full NAR forward
# ════════════════════════════════════════════════════════════════════
print('\n── D) torch.profiler: COMPILED Full NAR forward (torch.compile+CUDA Graphs, bs=1) ──')
_store_cp_full = _run_profiler(
    compiled_encoder, compiled_decoder, 'compiled_full_forward',
    'results/trace_compiled_full.json', 'results/profiler_compiled_full.txt',
    use_graphs=True)

# ── Kernel launch comparison: baseline vs compiled ────────────────────
def _launch_count(store):
    if not (DEVICE == 'cuda' and store.get('avgs')):
        return None
    e = next((e for e in store['avgs'] if e.key == 'cudaLaunchKernel'), None)
    return e.count if e is not None else None

_launches_bl = _launch_count(_store_bl_full)
_launches_cp = _launch_count(_store_cp_full)

print('\n── Kernel launch comparison (Full NAR forward, bs=1, 50 profiled spectra) ──')
if _launches_bl is not None:
    print(f'Baseline (eager)                : {_launches_bl/PROF_ACTIVE:.0f} launches/spectrum')
if _launches_cp is not None:
    print(f'Compiled (torch.compile+Graphs) : {_launches_cp/PROF_ACTIVE:.0f} launches/spectrum')
if _launches_bl and _launches_cp:
    print(f'Reduction factor                : {_launches_bl/max(_launches_cp,1):.1f}×')

Pre-fetching 70 bs=1 batches for profiler…


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

Using 70 batches  (truncated to N_PEAKS: 0)

── A) torch.profiler: BASELINE SpectrumEncoder (eager, bs=1) ──
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.66%       5.762ms       100.00%     868.563ms      17.371ms       0.000us         0.00%      56.890ms       1.138ms            50  
                                       baseline_encoder        14.00%     121.578ms        99.20% 

In [12]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 9 — Synthetic Micro-benchmark: Baseline vs Compiled (bs=1, 20 reps)
# FIX 1: cudagraph_mark_step_begin() before every encoder call that's
#        followed by a decoder call (same root cause as Cell 7/8).
# FIX 2: the encoder output held for the decoder-only timing loop is now
#        .clone()-d immediately — without this, the 20 repeated calls in
#        the encoder-only loop overwrite that exact CUDA-Graph buffer
#        before the decoder-only loop tries to read it.
# ═══════════════════════════════════════════════════════════════════════
def make_synth_batch(bs=1, device=DEVICE):
    n_real = 123
    mzs_s  = torch.zeros(bs, N_PEAKS, device=device)
    ints_s = torch.zeros(bs, N_PEAKS, device=device)
    for i in range(bs):
        mzs_s[i, :n_real]  = torch.rand(n_real, device=device) * 1303 + 301
        ints_s[i, :n_real] = torch.rand(n_real, device=device)
        norm = ints_s[i, :n_real].norm().clamp(min=1e-8)
        ints_s[i, :n_real] /= norm
    charge = 2.0; pmz = 600.0
    precs  = torch.tensor(
        [[(pmz - 1.007276) * charge, charge, pmz]] * bs,
        dtype=torch.float, device=device)
    return mzs_s, ints_s, precs

smzs, sints, sprecs = make_synth_batch(bs=1)
szero = torch.zeros((1, model.max_peptide_len), dtype=torch.long, device=DEVICE)

def _mark_step():
    if DEVICE == 'cuda':
        torch.compiler.cudagraph_mark_step_begin()

def _time_pair(encoder_fn, decoder_fn, n_reps=20, n_warm=5, use_graphs=False):
    def _mark():
        if use_graphs:
            _mark_step()

    with torch.no_grad():
        for _ in range(n_warm):
            _mark()
            _me, _mk = encoder_fn(smzs, sints)
            decoder_fn(tokens=szero, memory=_me,
                      memory_key_padding_mask=_mk, precursors=sprecs)
    _sync()

    # Fresh, CLONED encoder output for the decoder-only loop below.
    # Cloning detaches it from the CUDA-Graph static output buffer so it
    # stays valid even after encoder_fn() is called again in the
    # encoder-only timing loop further down.
    with torch.no_grad():
        _mark()
        _me, _mk = encoder_fn(smzs, sints)
        _me = _me.clone()
        _mk = _mk.clone() if _mk is not None else None
    _sync()

    _et, _dt, _ft = [], [], []
    with torch.no_grad():
        for _ in range(n_reps):
            _mark()
            _sync(); t0 = time.perf_counter()
            encoder_fn(smzs, sints)
            _sync(); _et.append((time.perf_counter() - t0) * 1000)
        for _ in range(n_reps):
            _sync(); t0 = time.perf_counter()
            decoder_fn(tokens=szero, memory=_me,
                      memory_key_padding_mask=_mk, precursors=sprecs)
            _sync(); _dt.append((time.perf_counter() - t0) * 1000)
        for _ in range(n_reps):
            _mark()
            _sync(); t0 = time.perf_counter()
            _me2, _mk2 = encoder_fn(smzs, sints)
            decoder_fn(tokens=szero, memory=_me2,
                      memory_key_padding_mask=_mk2, precursors=sprecs)
            _sync(); _ft.append((time.perf_counter() - t0) * 1000)

    return (float(np.mean(_et[5:])), float(np.mean(_dt[5:])), float(np.mean(_ft[5:])))

baseline_enc_ms, baseline_dec_ms, baseline_full_ms = _time_pair(
    model.encoder, model.decoder, use_graphs=False)
compiled_enc_ms, compiled_dec_ms, compiled_full_ms = _time_pair(
    compiled_encoder, compiled_decoder, use_graphs=True)

baseline_synth_tp = 1000.0 / baseline_full_ms
compiled_synth_tp = 1000.0 / compiled_full_ms

print(f'\n── Synthetic Micro-timings (20 reps, drop first 5, bs=1) ──')
print(f'{"Metric":<30} {"Baseline":>12} {"Compiled":>12} {"Speedup":>9}')
print('-' * 65)
_rows = [
    ('SpectrumEncoder (ms)', baseline_enc_ms,  compiled_enc_ms),
    ('NAR Decoder (ms)',     baseline_dec_ms,  compiled_dec_ms),
    ('Full forward (ms)',    baseline_full_ms, compiled_full_ms),
    ('Throughput (spec/s)',  baseline_synth_tp, compiled_synth_tp),
]
for label, bv, cv in _rows:
    spd = (bv / max(cv, 0.001)) if 'Throughput' not in label else (cv / max(bv, 0.001))
    print(f'  {label:<28} {bv:>12.2f} {cv:>12.2f} {spd:>8.2f}×')

pd.DataFrame({
    'metric'  : ['enc_ms', 'dec_ms', 'full_ms', 'throughput_spec_s'],
    'baseline': [baseline_enc_ms, baseline_dec_ms, baseline_full_ms, baseline_synth_tp],
    'compiled': [compiled_enc_ms, compiled_dec_ms, compiled_full_ms, compiled_synth_tp],
    'speedup' : [baseline_enc_ms/max(compiled_enc_ms,0.001),
                 baseline_dec_ms/max(compiled_dec_ms,0.001),
                 baseline_full_ms/max(compiled_full_ms,0.001),
                 compiled_synth_tp/max(baseline_synth_tp,0.001)],
}).to_csv('results/nar_baseline_vs_compiled_synthetic.csv', index=False)
print('\nSaved: results/nar_baseline_vs_compiled_synthetic.csv')


── Synthetic Micro-timings (20 reps, drop first 5, bs=1) ──
Metric                             Baseline     Compiled   Speedup
-----------------------------------------------------------------
  SpectrumEncoder (ms)                 8.81         8.53     1.03×
  NAR Decoder (ms)                    15.78        18.50     0.85×
  Full forward (ms)                   24.29        29.57     0.82×
  Throughput (spec/s)                 41.17        33.81     0.82×

Saved: results/nar_baseline_vs_compiled_synthetic.csv


In [13]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 10 — Plots + Summary: Baseline NAR vs Compiled NAR
# ═══════════════════════════════════════════════════════════════════════
import datetime

b1 = timing_baseline[1]
c1 = timing_compiled[1]

_lats_bl = [timing_baseline[bs]['total_mean'] for bs in BATCH_SIZES]
_lats_cp = [timing_compiled[bs]['total_mean'] for bs in BATCH_SIZES]
_tps_bl  = [timing_baseline[bs]['throughput'] for bs in BATCH_SIZES]
_tps_cp  = [timing_compiled[bs]['throughput'] for bs in BATCH_SIZES]
_xi   = list(range(len(BATCH_SIZES)))
_xlbl = [str(b) for b in BATCH_SIZES]

# ── Figure 1 — Stage breakdown ────────────────────────────────────────
fig1, ax1 = plt.subplots(1, 2, figsize=(14, 5))
fig1.suptitle('Stage Breakdown (bs=1) — Baseline NAR vs Compiled NAR', fontweight='bold')

_stages  = ['Fetch', 'H2D+pad', 'Encoder', 'Decoder', 'Write']
_bl_vals = [b1['fetch_mean'], b1['h2d_mean'], b1['enc_mean'], b1['nar_mean'], b1['write_mean']]
_cp_vals = [c1['fetch_mean'], c1['h2d_mean'], c1['enc_mean'], c1['nar_mean'], c1['write_mean']]
_ymax = max(max(_bl_vals), max(_cp_vals)) * 1.25

for ax, vals, color, title in [
    (ax1[0], _bl_vals, '#D85A30', f'Baseline (eager)\nTotal: {b1["total_mean"]:.1f} ms/spec'),
    (ax1[1], _cp_vals, '#1D9E75', f'Compiled (torch.compile + CUDA Graphs)\nTotal: {c1["total_mean"]:.1f} ms/spec'),
]:
    bars = ax.bar(_stages, vals, color=color, edgecolor='none', width=0.55)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + _ymax * 0.02, f'{v:.2f}', ha='center', fontsize=9)
    ax.set_title(title); ax.set_ylabel('ms / spectrum')
    ax.set_ylim(0, _ymax)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('results/nar_stage_baseline_vs_compiled.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/nar_stage_baseline_vs_compiled.png')

# ── Figure 2 — Throughput & Latency vs Batch Size ─────────────────────
fig2, ax2 = plt.subplots(1, 2, figsize=(14, 5))
fig2.suptitle('NAR Performance vs Batch Size — Baseline vs Compiled', fontweight='bold')

ax2[0].plot(_xi, _tps_bl, 'o-', color='#D85A30', lw=2, ms=8, label='Baseline (eager)')
ax2[0].plot(_xi, _tps_cp, 's-', color='#1D9E75', lw=2, ms=8, label='Compiled (torch.compile+Graphs)')
for i, (yb, yc) in enumerate(zip(_tps_bl, _tps_cp)):
    ax2[0].text(i, yb + max(_tps_cp) * 0.02, f'{yb:.0f}', ha='center', fontsize=8, color='#D85A30')
    ax2[0].text(i, yc + max(_tps_cp) * 0.06, f'{yc:.0f}', ha='center', fontsize=8, color='#1D9E75')
ax2[0].set_xticks(_xi); ax2[0].set_xticklabels(_xlbl)
ax2[0].set_xlabel('Batch size'); ax2[0].set_ylabel('Throughput (spec/s)')
ax2[0].set_title('Throughput vs Batch Size')
ax2[0].legend(fontsize=8, frameon=False)
ax2[0].spines[['top', 'right']].set_visible(False)

ax2[1].plot(_xi, _lats_bl, 'o-', color='#D85A30', lw=2, ms=8, label='Baseline (eager)')
ax2[1].plot(_xi, _lats_cp, 's-', color='#1D9E75', lw=2, ms=8, label='Compiled (torch.compile+Graphs)')
ax2[1].axhline(35, color='#9B59B6', lw=1.5, ls='--', label='35 ms (~29 Hz)')
ax2[1].axhline(50, color='#E67E22', lw=1.2, ls='-.', label='50 ms (20 Hz)')
ax2[1].axhline(10, color='black',   lw=1.5, ls=':',  label='10 ms (100 Hz target)')
for i, (yb, yc) in enumerate(zip(_lats_bl, _lats_cp)):
    ax2[1].text(i, yb + max(_lats_bl) * 0.02, f'{yb:.1f}', ha='center', fontsize=8, color='#D85A30')
    ax2[1].text(i, yc + max(_lats_bl) * 0.06, f'{yc:.1f}', ha='center', fontsize=8, color='#1D9E75')
ax2[1].set_xticks(_xi); ax2[1].set_xticklabels(_xlbl)
ax2[1].set_xlabel('Batch size'); ax2[1].set_ylabel('ms / spectrum')
ax2[1].set_title('Latency per Spectrum vs Batch Size')
ax2[1].legend(fontsize=8, frameon=False)
ax2[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('results/nar_performance_baseline_vs_compiled.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/nar_performance_baseline_vs_compiled.png')

# ── Figure 3 — Synthetic comparison + latency distribution overlay ────
fig3, ax3 = plt.subplots(1, 2, figsize=(14, 5))
fig3.suptitle('Synthetic Timing + Real Latency Distribution — Baseline vs Compiled', fontweight='bold')

_spd_synth = baseline_full_ms / max(compiled_full_ms, 0.001)
bars3 = ax3[0].bar(['Baseline\n(eager)', 'Compiled\n(torch.compile+Graphs)'],
                    [baseline_full_ms, compiled_full_ms],
                    color=['#D85A30', '#1D9E75'], edgecolor='none', width=0.35)
for b, v, tp in zip(bars3, [baseline_full_ms, compiled_full_ms], [baseline_synth_tp, compiled_synth_tp]):
    ax3[0].text(b.get_x() + b.get_width() / 2, b.get_height() + max(baseline_full_ms, compiled_full_ms) * 0.02,
                f'{v:.1f} ms\n{tp:.1f} spec/s', ha='center', fontsize=10)
ax3[0].axhline(10, color='black', lw=1.5, ls=':', label='10 ms (100 Hz target)')
ax3[0].set_title(f'Full Forward Pass (synthetic, bs=1)\nSpeedup: {_spd_synth:.2f}×')
ax3[0].set_ylabel('ms / spectrum')
ax3[0].legend(fontsize=9, frameon=False)
ax3[0].spines[['top', 'right']].set_visible(False)

_raw_bl = timing_baseline[1]['raw']['total']
_raw_cp = timing_compiled[1]['raw']['total']
ax3[1].hist(_raw_bl, bins=25, color='#D85A30', alpha=0.55, label=f'Baseline (mean {np.mean(_raw_bl):.1f}ms)')
ax3[1].hist(_raw_cp, bins=25, color='#1D9E75', alpha=0.55, label=f'Compiled (mean {np.mean(_raw_cp):.1f}ms)')
ax3[1].axvline(10, color='black', lw=1.5, ls=':', alpha=0.6, label='10 ms (100 Hz target)')
ax3[1].legend(fontsize=8, frameon=False)
ax3[1].set_title(f'Real-Spectra Latency Distribution (bs=1, {len(_raw_bl)} spectra)')
ax3[1].set_xlabel('ms / spectrum')
ax3[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('results/nar_baseline_vs_compiled_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/nar_baseline_vs_compiled_comparison.png')

# ── Text Summary ────────────────────────────────────────────────────
_now = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')

_total_bl = b1['total_mean']; _total_cp = c1['total_mean']
_t10_bl = 'MEETS ✓' if _total_bl <= 10 else f'FAILS ({_total_bl:.1f} ms, {_total_bl/10:.1f}× over)'
_t10_cp = 'MEETS ✓' if _total_cp <= 10 else f'FAILS ({_total_cp:.1f} ms, {_total_cp/10:.1f}× over)'
_speedup_real_bs1 = _total_bl / max(_total_cp, 0.001)

summary = f"""CASANOVO NAR PROFILING — Baseline (PR #548 patch) vs torch.compile + CUDA Graphs
Generated  : {_now}
Hardware   : {GPU_NAME} | {TOTAL_VRAM:.1f} GB VRAM | PyTorch {torch.__version__}
Dataset    : {SUBSET_MGF} ({N_SUBSET} spectra) | Timed: {N_TIMING_SPECTRA} spectra per batch size
Batch sizes: {BATCH_SIZES}
Fixed peak budget (CUDA Graph requirement): N_PEAKS={N_PEAKS}
  (applied identically to both baseline and compiled runs for a fair comparison)

OPTIMIZATION APPLIED
  torch.compile(model.encoder, mode='reduce-overhead')
  torch.compile(model.decoder, mode='reduce-overhead')
  'reduce-overhead' = TorchInductor compilation + automatic CUDA Graph
  capture/replay, eliminating per-kernel CPU dispatch overhead.

REAL-SPECTRA STAGE BREAKDOWN (bs=1, {b1['n_spec']} spectra)
Baseline (eager):
{df_stage_baseline.to_string(index=False)}
  Throughput : {b1['throughput']:.1f} spec/s | 10ms target: {_t10_bl}

Compiled (torch.compile+CUDA Graphs):
{df_stage_compiled.to_string(index=False)}
  Throughput : {c1['throughput']:.1f} spec/s | 10ms target: {_t10_cp}

Speedup (bs=1, real spectra): {_speedup_real_bs1:.2f}×

MULTI-BATCH THROUGHPUT
Baseline:
{df_throughput_baseline.to_string(index=False)}

Compiled:
{df_throughput_compiled.to_string(index=False)}

SYNTHETIC MICRO-BENCHMARK (20 reps, bs=1)
  Encoder — baseline: {baseline_enc_ms:.2f} ms | compiled: {compiled_enc_ms:.2f} ms
  Decoder — baseline: {baseline_dec_ms:.2f} ms | compiled: {compiled_dec_ms:.2f} ms
  Full    — baseline: {baseline_full_ms:.2f} ms ({baseline_synth_tp:.1f} spec/s) | compiled: {compiled_full_ms:.2f} ms ({compiled_synth_tp:.1f} spec/s)
  Speedup (synthetic): {_spd_synth:.2f}×

GPU UTILIZATION
  Baseline : {gpu_util_mean_bl:.0f}% mean util | {gpu_vram_peak_bl:.2f} GB peak VRAM
  Compiled : {gpu_util_mean_cp:.0f}% mean util | {gpu_vram_peak_cp:.2f} GB peak VRAM

ONE-TIME COMPILE COST (excluded from all timed results above)
{pd.DataFrame([{'batch_size': bs, 'compile_seconds': round(timing_compiled[bs]['compile_s'],1)} for bs in BATCH_SIZES]).to_string(index=False)}

ARTIFACTS SAVED TO results/
  nar_stage_baseline_vs_compiled.png
  nar_performance_baseline_vs_compiled.png
  nar_baseline_vs_compiled_comparison.png
  trace_baseline_encoder.json | trace_baseline_full.json
  trace_compiled_encoder.json | trace_compiled_full.json   (ui.perfetto.dev)
  profiler_baseline_encoder.txt | profiler_baseline_full.txt
  profiler_compiled_encoder.txt | profiler_compiled_full.txt
  nar_baseline_stage_timing_bs1.csv | nar_baseline_throughput_all_bs.csv
  nar_compiled_stage_timing_bs1.csv | nar_compiled_throughput_all_bs.csv
  nar_baseline_vs_compiled_synthetic.csv
  nar_summary.txt   (this file)
"""

print(summary)
with open('results/nar_summary.txt', 'w') as fh:
    fh.write(summary)

print('\n── results/ ──')
for _f in sorted(os.listdir('results')):
    _fp = os.path.join('results', _f)
    print(f'  {_f:<55} {os.path.getsize(_fp)/1024:.1f} KB')
print('\nProfiling complete. Primary: results/nar_summary.txt')

Saved: results/nar_stage_baseline_vs_compiled.png
Saved: results/nar_performance_baseline_vs_compiled.png
Saved: results/nar_baseline_vs_compiled_comparison.png
CASANOVO NAR PROFILING — Baseline (PR #548 patch) vs torch.compile + CUDA Graphs
Generated  : 2026-06-21 14:18
Hardware   : NVIDIA L4 | 23.6 GB VRAM | PyTorch 2.7.1+cu128
Dataset    : subset_profile.mgf (6000 spectra) | Timed: 5000 spectra per batch size
Batch sizes: [1, 8, 32, 128, 512]
Fixed peak budget (CUDA Graph requirement): N_PEAKS=150
  (applied identically to both baseline and compiled runs for a fair comparison)

OPTIMIZATION APPLIED
  torch.compile(model.encoder, mode='reduce-overhead')
  torch.compile(model.decoder, mode='reduce-overhead')
  'reduce-overhead' = TorchInductor compilation + automatic CUDA Graph
  capture/replay, eliminating per-kernel CPU dispatch overhead.

REAL-SPECTRA STAGE BREAKDOWN (bs=1, 5000 spectra)
Baseline (eager):
               Stage  mean_ms  p50_ms  p95_ms
    DataLoader fetch    1.488  